In [100]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib
from sklearn import tree 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from lightgbm import LGBMClassifier 

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay

# Custom Functions
from credit_risk_modeling import model_eval
import importlib
importlib.reload(model_eval)

<module 'credit_risk_modeling.model_eval' from 'C:\\Users\\billy\\OneDrive\\Documents\\Finance_Projects\\credit_risk_modeling\\credit_risk_modeling\\model_eval.py'>

## Imports

In [101]:
X_train = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_train_tree.csv"
)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22686 entries, 0 to 22685
Data columns (total 8 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   numeric__person_age                         22686 non-null  float64
 1   numeric__person_income                      22686 non-null  float64
 2   numeric__person_emp_length                  22686 non-null  float64
 3   numeric__loan_amnt                          22686 non-null  float64
 4   numeric__loan_int_rate                      22686 non-null  float64
 5   numeric__loan_percent_income                22686 non-null  float64
 6   numeric__cb_person_cred_hist_length         22686 non-null  float64
 7   categorical__cb_person_default_on_file_1.0  22686 non-null  float64
dtypes: float64(8)
memory usage: 1.4 MB


In [102]:
X_test = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_test_tree.csv"
)
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9723 entries, 0 to 9722
Data columns (total 8 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   numeric__person_age                         9723 non-null   float64
 1   numeric__person_income                      9723 non-null   float64
 2   numeric__person_emp_length                  9723 non-null   float64
 3   numeric__loan_amnt                          9723 non-null   float64
 4   numeric__loan_int_rate                      9723 non-null   float64
 5   numeric__loan_percent_income                9723 non-null   float64
 6   numeric__cb_person_cred_hist_length         9723 non-null   float64
 7   categorical__cb_person_default_on_file_1.0  9723 non-null   float64
dtypes: float64(8)
memory usage: 607.8 KB


In [103]:
y_train = pd.read_csv(
    filepath_or_buffer = "../data/interim/y_train.csv"
)
y_train = y_train.values.ravel()

In [104]:
y_test = pd.read_csv(
    filepath_or_buffer = "../data/interim/y_test.csv"
)
y_test.head(1)
y_test = y_test.values.ravel()

## Compare Models

Not accouting for feature selection (unless), class imbalance, or hyperparameter tuning

In [143]:
untuned_models = [
    DummyClassifier(strategy="most_frequent"),
    tree.DecisionTreeClassifier(
        random_state = 42,
    ), # no sampling 
    AdaBoostClassifier(
        random_state=42
    ), # default estimator is DecisionTreeClassifier, default no sampling
    XGBClassifier(
        reg_lambda=0,
        reg_alpha=0,
        random_state=42
    ), # default embedded l2 regularization, default includes no sampling (but sampling is possible for both samples and features - would be pasting + subspacing so replacement=None)
    LGBMClassifier(
    objective='binary',
    reg_alpha = 0,
    reg_lambda = 0,
    random_state=42
    ), # default embedded l2 regularization, default includes no sampling (but sampling is possible for both samples and features - would be pasting + subspacing so replacement=None)
    HistGradientBoostingClassifier(
        l2_regularization= 0,
        random_state=42
    ), # no regularization, default no subsampling but can be turned on
    GradientBoostingClassifier(
        subsample = 1.0,
        random_state=42
    ), # default no subsampling regularization but Pasting can be turned on
    RandomForestClassifier(
        n_jobs=-1,
        random_state=42
    ), # default Bagging, Subspacing
    ExtraTreesClassifier(
        n_jobs=-1,
        random_state=42
    ), # default all samples but Bagging can be turned on, Subspacing for features
    BaggingClassifier(estimator=tree.DecisionTreeClassifier(
        random_state=42), 
        n_estimators=50, 
        bootstrap=True, 
        max_samples=1.0,
        max_features=1.0, 
        random_state=42),
    VotingClassifier(
        estimators= [('lgbm', LGBMClassifier(n_jobs=-1, random_state=42)), ('xgb', XGBClassifier(n_jobs=-1, random_state=42)), ('hgb', HistGradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(n_jobs=-1, random_state=42))],
        voting='soft',
        n_jobs=-1
    ),
    StackingClassifier(
        estimators= [('lgbm', LGBMClassifier(n_jobs=-1,random_state=42)), ('xgb', XGBClassifier(n_jobs=-1, random_state=42)), ('hgb', HistGradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(n_jobs=-1, random_state=42))],
        final_estimator= LGBMClassifier(random_state=42),
        n_jobs=-1
    ),
    StackingClassifier(
        estimators= [('lgbm', LGBMClassifier(n_jobs=-1,random_state=42)), ('xgb', XGBClassifier(n_jobs=-1, random_state=42)), ('hgb', HistGradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(n_jobs=-1, random_state=42))],
        final_estimator= StackingClassifier(
            estimators= [('lgbm', LGBMClassifier(n_jobs=-1,random_state=42)), ('xgb', XGBClassifier(n_jobs=-1, random_state=42)), ('hgb', HistGradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(n_jobs=-1, random_state=42))],
            final_estimator= LGBMClassifier(n_jobs=-1, random_state=42),
            n_jobs=-1
        ),
        n_jobs=-1
    ) # multiple layer
]

In [144]:

untuned_model_performance, fitted_models_internal = model_eval.comparing_models(untuned_models, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 4962, number of negative: 17724
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000761 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1061
[LightGBM] [Info] Number of data points in the train set: 22686, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.218725 -> initscore=-1.273111
[LightGBM] [Info] Start training from score -1.273111
[LightGBM] [Info] Number of positive: 4962, number of negative: 17724
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001148 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 866
[LightGBM] [Info] Number of data points in the train set: 22686, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.218725 -> initscore=-1.273111
[LightGBM] [Info] Start training from score -1.273111


c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 4962, number of negative: 17724
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000430 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 866
[LightGBM] [Info] Number of data points in the train set: 22686, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.218725 -> initscore=-1.273111
[LightGBM] [Info] Start training from score -1.273111


c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [199]:
untuned_model_performance = untuned_model_performance.sort_values(
    by='roc_auc',
    ascending=False,
    axis=0
)

In [200]:
untuned_model_performance['untuned_model'] = 1

## Hyperparameter Tuning, Class Imbalance, Feature Selection

### LGBMClassifier

In [171]:
lgbm = LGBMClassifier(
    objective='binary',
    unbalance = True,
    random_state=42
    )

In [172]:
lgbm_param_dist = {
    "num_leaves": stats.randint(16, 256),
    "min_data_in_leaf": stats.randint(10, 500),
    "max_depth": stats.randint(3, 10),
    "learning_rate": stats.loguniform(0.01, 0.2),
    "n_estimators": stats.randint(200, 1000),
    "feature_fraction": stats.uniform(0.6, 0.4),
    "bagging_fraction": stats.uniform(0.6, 0.4),
    "bagging_freq": stats.randint(1, 6),
    "lambda_l1": stats.loguniform(1e-3, 1),
    "lambda_l2" :stats.loguniform(1e-3, 1),
}

In [173]:
lgbm_tuned = HalvingRandomSearchCV(
estimator = lgbm,
param_distributions=lgbm_param_dist,
scoring = 'roc_auc',
n_jobs=-1,
random_state=42)

### XGBClassifier

In [174]:
xgbm = XGBClassifier(
    objective='binary:logistic',
    unbalance = True,
    random_state=42
    )

In [175]:
xgbm_param_dist = {
    "max_depth": stats.randint(3, 10),
    "min_child_weight": stats.loguniform(1e-1, 1e2),
    "gamma": stats.loguniform(1e-3, 1),
    "reg_alpha": stats.loguniform(1e-3, 10),
    "reg_lambda": stats.loguniform(1e-3, 10),
    "subsample": stats.uniform(0.5, 0.5),
    "colsample_bytree": stats.uniform(0.5, 0.5),
    "learning_rate": stats.loguniform(0.01, 0.3),
    "n_estimators": stats.randint(200, 1000),
}

In [176]:
xgbm_tuned = HalvingRandomSearchCV(
estimator = xgbm,
param_distributions=xgbm_param_dist,
scoring = 'roc_auc',
n_jobs=-1,
random_state=42)

### HistGradientBoosting

In [177]:
hgb = HistGradientBoostingClassifier(
    random_state=42, 
    class_weight='balanced')

In [178]:
hgb_param_dist = {
    "max_depth": stats.randint(3, 10),
    "max_leaf_nodes": stats.randint(15, 255),
    "min_samples_leaf": stats.randint(10, 200),
    "learning_rate": stats.loguniform(0.01, 0.2),
    "max_iter": stats.randint(200, 1000),
    "l2_regularization": stats.loguniform(1e-3, 1),
    "max_bins": stats.randint(64, 255),
}

In [179]:
hgb_tuned = HalvingRandomSearchCV(
estimator = hgb,
param_distributions=hgb_param_dist,
scoring = 'roc_auc',
n_jobs=-1,
random_state=42)

### Random Forest

In [180]:
rf = RandomForestClassifier(
    n_jobs=-1, 
    random_state=42, 
    class_weight='balanced')

In [181]:
rf_param_dist = {
    "max_depth": stats.randint(3, 20),
    "min_samples_split": stats.randint(2, 20),
    "min_samples_leaf": stats.randint(1, 10),
    "n_estimators": stats.randint(200, 1000),
    "max_features": stats.uniform(0.4, 0.6),
    "bootstrap": [True, False],
    "max_leaf_nodes": stats.randint(10, 200),
}

In [182]:
rf_tuned = HalvingRandomSearchCV(
estimator = rf,
param_distributions=rf_param_dist,
scoring = 'roc_auc',
n_jobs=-1,
random_state=42)

In [183]:
manually_tuned_models = [
    lgbm_tuned,
    xgbm_tuned,
    hgb_tuned,
    rf_tuned
]

In [184]:
manual_tuned_model_performance, fitted_models_manual = model_eval.comparing_manually_tuned_models(manually_tuned_models, X_train, y_train, X_test, y_test)

c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan ... nan nan nan]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan ... 0.5 0.5 0.5]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.77427169 0.76477588 0.79296043]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.8534995  0.80559915 0.82220108]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeli

[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Warning] feature_fraction is set=0.849362383263776, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.849362383263776
[LightGBM] [Warning] lambda_l1 is set=0.051066028531447934, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.051066028531447934
[LightGBM] [Warning] lambda_l2 is set=0.07277487019368942, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.07277487019368942
[LightGBM] [Warning] bagging_fraction is set=0.8938311692485652, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8938311692485652
[LightGBM] [Warning] bagging_freq is set=2, subsample_freq=0 will be ignored. Current value: bagging_freq=2
[LightGBM] [Warning] min_data_in_leaf is set=17, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=17
[LightGBM] [Warning] feature_fraction is set=0.849362383263776, colsample_by

c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan ... nan nan nan]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.50545455 0.77833333 0.54545455]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.79447258 0.78341992 0.7983338 ]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.82881641 0.83811325 0.8529846 ]
  warnings.warn(
c:\Use

In [201]:
manual_tuned_model_performance['untuned_model'] = 0

In [202]:
manual_tuned_model_performance = manual_tuned_model_performance.sort_values(
    ascending= False,
    by= 'roc_auc'
)

All the models performed worse by 1/1000th.

## Probability Calibration

In [206]:
model_performances = model_eval.combine_model_performance(untuned_model_performance, manual_tuned_model_performance)

In [207]:
model_performances

,model,roc_auc,pr_auc,log_loss,brier_score,matthews_corrcoef,untuned_model
10,VotingClassifier (n_jobs=-1),0.901203,0.777688,0.304055,0.091458,0.617737,1
11,StackingClassifier (n_jobs=-1),0.900587,0.775782,0.304807,0.091371,0.624813,1
4,"LGBMClassifier (n_jobs=None, class_weight=None)",0.898813,0.772406,0.306813,0.092587,0.614540,1
12,StackingClassifier (n_jobs=-1),0.897882,0.767502,0.308140,0.092669,0.621075,1
0,"LGBMClassifier (n_jobs=None, class_weight=None)",0.896230,0.768443,0.390904,0.121333,0.578003,0
3,XGBClassifier (n_jobs=None),0.893956,0.761042,0.318800,0.095769,0.601477,1
5,HistGradientBoostingClassifier (class_weight=N...,0.892686,0.764979,0.313513,0.094313,0.604079,1
9,BaggingClassifier (n_jobs=None),0.888014,0.756118,0.463621,0.095236,0.617986,1
2,HistGradientBoostingClassifier (class_weight=b...,0.887248,0.757069,0.403193,0.125000,0.574247,0
7,"RandomForestClassifier (n_jobs=-1, class_weigh...",0.885865,0.752596,0.357054,0.096286,0.598264,1
